In [1]:
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from tensor import Tensor
from nn import MLP

In [2]:
mnist = fetch_openml(
    "mnist_784",
    version=1,
    as_frame=False
)

X = mnist.data
y = mnist.target.astype(int)

In [3]:
print(X.shape)
print(y.shape)

(70000, 784)
(70000,)


In [4]:
model = MLP(784, [16, 16, 10])
n_samples = 10
lr = 0.01

X_train = X[:n_samples] / 255.0 # Normalization of pixel values
y_train = y[:n_samples]

In [5]:
def softmax(logits):
    # Subtract max per sample for numerical stability (prevents exp overflow).
    result = []
    for sample_logits in logits:
        max_logit = max(l.data for l in sample_logits)
        shifted = [l - max_logit for l in sample_logits]
        exp_logits = [s.exp() for s in shifted]
        exp_sum = sum(exp_logits)
        result.append([s / exp_sum for s in exp_logits])
    return result

def loss_fn():
    logits = model(X_train)
    probs = softmax(logits)
    # Cross entropy loss
    losses = [-probs_i[yi].log() for probs_i, yi in zip(probs, y_train)]
    return sum(losses) / len(losses)

for epoch in range(100):
    total_loss = loss_fn()

    model.zero_grad() # clear grads before each backward pass
    total_loss.backward()

    for p in model.parameters():
        p.data -= lr * p.grad # gradient descent step

    print("step:", epoch, "-> total loss:", total_loss.data)

step: 0 -> total loss: 17.77038144399328
step: 1 -> total loss: 13.403000433804117
step: 2 -> total loss: 9.974978580042338
step: 3 -> total loss: 7.5297764976837955
step: 4 -> total loss: 5.963980855007227
step: 5 -> total loss: 4.820716755612239
step: 6 -> total loss: 3.9256559665829025
step: 7 -> total loss: 3.1944481751475546
step: 8 -> total loss: 2.6110052421042997
step: 9 -> total loss: 2.1538405355318
step: 10 -> total loss: 1.812673723185369
step: 11 -> total loss: 1.5315631252576305
step: 12 -> total loss: 1.2829581942006767
step: 13 -> total loss: 1.057552095807082
step: 14 -> total loss: 0.8787994824554339
step: 15 -> total loss: 0.7436823667431279
step: 16 -> total loss: 0.6290105735608015
step: 17 -> total loss: 0.5363265853385651
step: 18 -> total loss: 0.4686327670870425
step: 19 -> total loss: 0.42382873145754235
step: 20 -> total loss: 0.38770642379637577
step: 21 -> total loss: 0.3566153514584236
step: 22 -> total loss: 0.33011061579866485
step: 23 -> total loss: 0.3

In [6]:
X_test = X[n_samples: 10+n_samples] / 255.0 # Normalization of pixel values
y_test = y[n_samples: 10+n_samples]

def infer(X_test, y_test):
    logits = model(X_test)
    probs = softmax(logits)

    y_pred = []
    for probs_i in probs:
        y_pred.append(max(range(len(probs_i)), key=lambda i: probs_i[i].data))

    return y_pred

result = infer(X_test, y_test)

In [7]:
print(y_test)
print(result)

[3 5 3 6 1 7 2 8 6 9]
[2, 3, 4, 5, 1, 9, 4, 5, 5, 5]
